# Chronos forecast caches (CUDA)

This notebook lives in `scripts/`, next to `run_chronos_cache.py`. Open it from the repository root or from `scripts/`. A flat upload still works if this file sits beside `run_chronos_cache.py` and `chronos_input_y.parquet`.

**Restart the kernel, then Run All.** A kernel that already imported the server's Chronos will keep that old copy.

The KnoxHub Chronos build calls `T5Stack` in a way transformers 5 no longer accepts. The install cell puts Chronos 2.3.2 in `/tmp/chronos_pkg` and loads that copy first. It does not upgrade the server's `pandas` or replace the CUDA `torch`.

Chronos is zero-shot here: these cells run inference and write `.npz` caches. After you download them into `data/features/chronos/` on your Mac, training and evaluation are:

```bash
python scripts/run_early_warning_experiment.py
```

A cache that already exists is skipped. Set `SMOKE = True` in the next cell for a 64-origin check first.

In [ ]:
from pathlib import Path

SMOKE = False       # True: 64 origins per cache, written to ./chronos_smoke/
BATCH_SIZE = None   # None uses 256 for Bolt and 32 for T5; set 16 if CUDA runs out of memory
DEVICE = "cuda"

JOBS = [
    ("bolt-small", 512),   # primary
    ("bolt-small", 256),   # context-length sensitivity
    ("bolt-small", 1024),
    ("bolt-base", 512),    # model-size sensitivity
    ("t5-small", 512),     # capstone checkpoint, 20 sample paths
]

def script_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, start / "scripts", *start.parents):
        if (candidate / "run_chronos_cache.py").exists():
            return candidate
    raise SystemExit(f"run_chronos_cache.py not found from {start}")

FOLDER = script_dir()
print("folder:", FOLDER)

In [ ]:
import subprocess
import sys

target = "/tmp/chronos_pkg"
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--target", target, "--upgrade", "--no-deps",
    "chronos-forecasting==2.3.2",
])
sys.path.insert(0, target)

import torch
import chronos

assert torch.cuda.is_available(), "CUDA is not available in this kernel. Switch to a GPU kernel, restart, and rerun."
loaded = Path(chronos.__file__).resolve()
assert str(loaded).startswith(target), f"Still using the server Chronos at {loaded}. Restart the kernel and rerun."
print(f"chronos {chronos.__version__}  torch {torch.__version__}  {torch.cuda.get_device_name(0)}")
print("chronos loaded from", loaded.parent)

In [ ]:
sys.path.insert(0, str(FOLDER))
from run_chronos_cache import run

candidates = [
    FOLDER / "chronos_input_y.parquet",
    FOLDER / "data" / "features" / "chronos_input_y.parquet",
    FOLDER.parent / "data" / "features" / "chronos_input_y.parquet",
]
src = next((p for p in candidates if p.exists()), None)
if src is None:
    raise FileNotFoundError(f"chronos_input_y.parquet not found in {FOLDER}")

out_dir = FOLDER / ("chronos_smoke" if SMOKE else "chronos")
out_dir.mkdir(exist_ok=True)
limit = 64 if SMOKE else None
print("input:", src)
print("output:", out_dir)

for variant, context in JOBS:
    dest = out_dir / f"{variant}_ctx{context}.npz"
    if dest.exists():
        print(f"skip {dest.name} (already exists)", flush=True)
        continue
    print(f"=== {variant} context={context} -> {dest}", flush=True)
    run(
        src, dest, variant, context,
        horizon=144, every_hours=6, start="2016-02-01T00:00Z",
        device=DEVICE,
        batch_size=BATCH_SIZE or (32 if variant.startswith("t5") else 256),
        num_samples=20, seed=0, limit=limit,
    )

print("\nfinished. Download these files:")
for p in sorted(out_dir.glob("*.npz")):
    print(f"  {p}  ({p.stat().st_size / 1e6:.1f} MB)")
print("\nPut them in data/features/chronos/ on your Mac, then run")
print("  python scripts/run_early_warning_experiment.py")